# 04 — Evaluate (mAP) & Build Submission

**Inputs:** 00 (data), 02/03 (trained checkpoints), + your code Dataset.

1. Validates on the fully labeled test set with COCO-style mAP
   (same metric as the Kaggle leaderboard).
2. Inspects `sample_submission.csv` and writes predictions in that format.

To attach checkpoints from 02/03, add those notebooks' saved versions as
Inputs too (Add Input → Your Work → select notebook version).

In [ ]:
import glob, sys
from pathlib import Path

hits = glob.glob('/kaggle/input/**/train_dataset.json', recursive=True)
ANN = Path(hits[0]).parent
IMG = ANN.parent / 'images'
if not IMG.exists():
    IMG = ANN / 'images'
SRC = glob.glob('/kaggle/input/**/src', recursive=True)
if SRC: sys.path.insert(0, str(Path(SRC[0]).parent))
else: sys.path.insert(0, '..')

# Find available checkpoints (baseline and/or PU ablation)
CKPTS = sorted(glob.glob('/kaggle/input/**/model_*.pth', recursive=True))
print('Annotations:', ANN, '\nCheckpoints found:')
for c in CKPTS: print(' ', c)

In [ ]:
import torch
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from torch.utils.data import DataLoader

from src.data.coco_dataset import CocoPuDataset, collate_fn
from src.models.detector import build_model
from src.training.utils import eval_transforms

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
coco_gt = COCO(str(ANN / 'test_dataset.json'))
ds = CocoPuDataset(IMG, ANN / 'test_dataset.json', transforms=eval_transforms())
loader = DataLoader(ds, batch_size=4, shuffle=False, collate_fn=collate_fn)
cat_ids = sorted(c['id'] for c in coco_gt.loadCats(coco_gt.getCatIds()))

@torch.no_grad()
def run_model(ckpt_path, score_thr=0.05):
    model = build_model('faster_rcnn', num_classes=33)
    model.load_state_dict(torch.load(ckpt_path, map_location='cpu')['model'])
    model.to(device).eval()
    results = []
    for images, targets in loader:
        outputs = model([i.to(device) for i in images])
        for out, tgt in zip(outputs, targets):
            img_id = int(tgt['image_id'].item())
            for box, label, score in zip(out['boxes'].cpu(), out['labels'].cpu(), out['scores'].cpu()):
                if score < score_thr: continue
                x1, y1, x2, y2 = box.tolist()
                results.append({'image_id': img_id,
                                'category_id': cat_ids[int(label) - 1],
                                'bbox': [x1, y1, x2 - x1, y2 - y1],
                                'score': float(score)})
    return results

In [ ]:
# Evaluate each checkpoint -> mAP comparison table (paper results table)
import pandas as pd

rows = []
for ckpt in CKPTS:
    print('\nEvaluating', Path(ckpt).name)
    res = run_model(ckpt)
    if not res:
        print('  (no predictions)'); continue
    ev = COCOeval(coco_gt, coco_gt.loadRes(res), 'bbox')
    ev.evaluate(); ev.accumulate()
    ev.summarize()
    rows.append({'checkpoint': Path(ckpt).name,
                 'mAP@[.5:.95]': ev.stats[0], 'mAP@.5': ev.stats[1]})
    df_res = pd.DataFrame(res)
    df_res.to_csv(f"/kaggle/working/{Path(ckpt).stem}_preds.csv", index=False)
print(pd.DataFrame(rows))

In [ ]:
# Inspect the official sample submission format
import pandas as pd
sub_hits = glob.glob('/kaggle/input/**/sample_submission.csv', recursive=True)
if sub_hits:
    sample = pd.read_csv(sub_hits[0])
    print('Columns:', list(sample.columns))
    print(sample.head())
else:
    print('sample_submission.csv not attached - grab it from the Kaggle Data tab.')